# 31_body_language_probe

## Note: where the labels for each class come from

Planning Stage: 

The decoder predicts the team's 5 categories directly so the 8-class body list and the 8-to-5 matrix in notebook 30 are not used here.

The decoder only needs an emotion label per image or clip. The keypoints come from our own YOLO11m-pose encoder, so the label datasets do not need keypoints.

| Category | Labels from | Notes |
|---|---|---|
| alert_aggressive | Kaggle *angry*, Roboflow *aggressive*, Roboflow *fear* | fear follows the team taxonomy (fearful/defensive is an alert_aggressive subtype) |
| playful_excited | Kaggle *happy*, Roboflow *happy* | |
| distressed_lonely | Kaggle *sad*, Roboflow *anxiety* | anxious/whining is a distressed_lonely subtype |
| attention_seeking | **none** | needs another dataset or our own labelled clips |
| other | Kaggle *relaxed* | calm / neutral |

Sources: Kaggle Dog Emotion (~4,000 images: angry / happy / relaxed / sad), Roboflow Dog Emotion Videos (~1,873 frames: aggressive / anxiety / fear / happy).

Baseline: always predict the most common category in the training set.

Pending Datasets:
attention_seeking examples.

### Loading the Datasets from kaggle and Roboflow here

##### Steps/Psuedocode
- make a dataframe of 5 columsn labeled path of image, source(kaggle/roboflow), original_label, category as per 5 classes above, group
- Loop through all each dataset to get build the dataframe
- Need to do some kind of foramatting change for robotflow dataset

In [ ]:
# starting with the imports, will grow as needed
from pathlib import Path
from woofdata.path import get_data_dir

import pandas as pd
import numpy as np
import torch as t
import json

#### First is Kaggle since its simpler

In [ ]:
## making datafram with 5 columsn with first row as the header
## the header will have the following columns: "path to image", "source", "original_label", "category", "group"
df = pd.DataFrame(columns=["path to image", "source", "original_label", "category", "group"])

root = get_data_dir() / "raw" 
kaggle_root = get_data_dir() / "raw" / "dog_emotion_kaggle" / "Dog Emotion"
for folder in kaggle_root.iterdir():
    if folder.name in ["angry"]:
        for file in folder.iterdir():
            df = pd.concat([df, pd.DataFrame([[str(file.relative_to(root)), "kaggle", folder.name, "alert_aggressive", f"kaggle_{file.name}"]], columns=df.columns)], ignore_index=True)

    if folder.name in ["happy"]:
        for file in folder.iterdir():
            df = pd.concat([df, pd.DataFrame([[str(file.relative_to(root)), "kaggle", folder.name, "playful_excited", f"kaggle_{file.name}"]], columns=df.columns)], ignore_index=True)

    if folder.name in ["sad"]:
        for file in folder.iterdir():
            df = pd.concat([df, pd.DataFrame([[str(file.relative_to(root)), "kaggle", folder.name, "distressed_lonely", f"kaggle_{file.name}"]], columns=df.columns)], ignore_index=True)

    if folder.name in ["relaxed"]:
        for file in folder.iterdir():
            df = pd.concat([df, pd.DataFrame([[str(file.relative_to(root)), "kaggle", folder.name, "other", f"kaggle_{file.name}"]], columns=df.columns)], ignore_index=True)


roboflow_root = get_data_dir() / "raw" / "rf_dog_emotion"

df

#### Then Roboflow

In [ ]:
import json

rf_root = root / "rf_dog_emotion"

with open(rf_root / "train" / "_annotations.coco.json") as f:
    coco = json.load(f)

print(coco["categories"])
print(coco["images"][0])
print(coco["annotations"][0])


In [ ]:
id_to_label = {}
for cat in coco["categories"]:
    id_to_label[cat["id"]] = cat["name"]

print(id_to_label)


In [ ]:
image_to_label = {}
for ann in coco["annotations"]:
    image_to_label[ann["image_id"]] = id_to_label[ann["category_id"]]

print(list(image_to_label.items())[:5])


In [ ]:
rf_root = root / "rf_dog_emotion"
rf_map = {"Aggressive": "alert_aggressive", "Fear": "alert_aggressive", "Anxiety": "distressed_lonely"}

rows = []
for split in ["train", "valid", "test"]:
    coco = json.load(open(rf_root / split / "_annotations.coco.json"))
    cats = {c["id"]: c["name"] for c in coco["categories"]} # made a dict of key(id) and values(catergory)
    labels = {a["image_id"]: cats[a["category_id"]] for a in coco["annotations"]} # again a dict which says which image has which category
    for img in coco["images"]:
        label = labels.get(img["id"]) # getting emotion from labels dict
        if label in rf_map:
            file = rf_root / split / img["file_name"]
            video = img["file_name"].split("_mp4")[0].split("_jpg")[0]
            rows.append([str(file.relative_to(root)), "roboflow", label, rf_map[label], f"roboflow_{video}"])

df = pd.concat([df, pd.DataFrame(rows, columns=df.columns)], ignore_index=True)
df


### Baseline

##### Is just the most frequent label in the combined database and thats what I compare the model to since that seems like comparing with the mean of the dataset

In [ ]:
# print the most common original_label and category in the dataframe
SBR = df["category"].value_counts()
print("Most common category:", SBR.idxmax(), "with count:", SBR.max())

### Now we have to run the encoder on the combined dataset created

##### steps/psuedocode
- load the pretrained model that was fined tuned for dogs
- Loop over each image in the dataset to label dogs and keypoints in the dataset

In [ ]:
from ultralytics import YOLO
model = YOLO("../../run_outputs/yolo11m-pose_full/best.pt")
sample = df.sample(5, random_state=42)

for path in sample["path to image"]:
    img_path = root / path
    r = model.predict(img_path, device="mps", verbose=False)[0]
    kp = r.keypoints.xy.cpu().numpy()
    print(r.boxes)
    print(r.boxes.xyxy)
    print(r.boxes.conf)
    print(kp)
    print(len(r.boxes))
    